In [2]:

import numpy as np
import matplotlib.pyplot as plt
from helpers import *
from mse_gd import *
from mse_sgd import *
import importlib, helpers
importlib.reload(helpers)
from helpers import standardize, build_model_data

In [3]:
x_train, x_test, y_train, train_ids, test_ids = load_csv_data("dataset")

In [4]:
y_train.shape, x_train.shape, x_train.shape

((328135,), (328135, 321), (328135, 321))

In [5]:
def load_feature_names(data_path):
    with open(os.path.join(data_path, "x_train.csv"), "r") as f:
        header = f.readline().strip().split(",")  # read first line
    return header[1:]  # skip the "Id" column

In [6]:
feature_names = load_feature_names("dataset")
#print(feature_names)
feature_to_index = {name: i for i, name in enumerate(feature_names)}
print(feature_to_index)

{'_STATE': 0, 'FMONTH': 1, 'IDATE': 2, 'IMONTH': 3, 'IDAY': 4, 'IYEAR': 5, 'DISPCODE': 6, 'SEQNO': 7, '_PSU': 8, 'CTELENUM': 9, 'PVTRESD1': 10, 'COLGHOUS': 11, 'STATERES': 12, 'CELLFON3': 13, 'LADULT': 14, 'NUMADULT': 15, 'NUMMEN': 16, 'NUMWOMEN': 17, 'CTELNUM1': 18, 'CELLFON2': 19, 'CADULT': 20, 'PVTRESD2': 21, 'CCLGHOUS': 22, 'CSTATE': 23, 'LANDLINE': 24, 'HHADULT': 25, 'GENHLTH': 26, 'PHYSHLTH': 27, 'MENTHLTH': 28, 'POORHLTH': 29, 'HLTHPLN1': 30, 'PERSDOC2': 31, 'MEDCOST': 32, 'CHECKUP1': 33, 'BPHIGH4': 34, 'BPMEDS': 35, 'BLOODCHO': 36, 'CHOLCHK': 37, 'TOLDHI2': 38, 'CVDSTRK3': 39, 'ASTHMA3': 40, 'ASTHNOW': 41, 'CHCSCNCR': 42, 'CHCOCNCR': 43, 'CHCCOPD1': 44, 'HAVARTH3': 45, 'ADDEPEV2': 46, 'CHCKIDNY': 47, 'DIABETE3': 48, 'DIABAGE2': 49, 'SEX': 50, 'MARITAL': 51, 'EDUCA': 52, 'RENTHOM1': 53, 'NUMHHOL2': 54, 'NUMPHON2': 55, 'CPDEMO1': 56, 'VETERAN3': 57, 'EMPLOY1': 58, 'CHILDREN': 59, 'INCOME2': 60, 'INTERNET': 61, 'WEIGHT2': 62, 'HEIGHT3': 63, 'PREGNANT': 64, 'QLACTLM2': 65, 'USEEQUI

In [21]:
# keeping just the features we want
features_to_keep = [26, 30, 32, 34, 35, 38, 39, 40, 43, 44, 45, 46, 47, 48, 50,52, 57, 58, 60,62, 63, 72, 73, 75, 76, 77, 82, 83, 84, 85, 86, 87, 89, 104, 136, 142, 144, 155, 156, 192, 193, 234, 235, 238, 242, 248, 250, 251, 252, 253, 259, 264, 276, 277, 301]
x_train = x_train[:, features_to_keep]
x_test = x_test[:, features_to_keep]
print(x_train.shape)

IndexError: index 57 is out of bounds for axis 1 with size 55

In [8]:
# Replace outliers in x_train with NaNs using the IQR method
Q1 = np.nanpercentile(x_train, 25, axis=0)
Q3 = np.nanpercentile(x_train, 75, axis=0)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = (x_train < lower_bound) | (x_train > upper_bound)
x_train_clean = x_train.copy()
x_train_clean[outliers] = np.nan

In [11]:
import os

# Create output folder for boxplots
boxplot_dir = "boxplots"
os.makedirs(boxplot_dir, exist_ok=True)

for i, feature_idx in enumerate(range(x_train_clean.shape[1])):
    feature_name = feature_names[features_to_keep[i]]
    data = x_train_clean[:, feature_idx]
    # General boxplot
    data_general = data[~np.isnan(data)]
    # Boxplot for y_train == -1
    data_neg1 = data[(y_train == -1) & (~np.isnan(data))]
    # Boxplot for y_train == 1
    data_pos1 = data[(y_train == 1) & (~np.isnan(data))]

    plt.figure(figsize=(8, 6))
    plt.boxplot([data_general, data_neg1, data_pos1], tick_labels=["All", "y=-1", "y=1"], showfliers=False)
    plt.title(feature_name)
    plt.ylabel("Value")
    plt.tight_layout()
    plt.savefig(os.path.join(boxplot_dir, f"{feature_name}.png"))
    plt.close()

In [12]:
# Divide features in x_train_clean into continuous and categorical

continuous_features = []
categorical_features = []

for idx, feature_idx in enumerate(features_to_keep):
    name = feature_names[feature_idx]
    col = x_train_clean[:, idx]
    unique_vals = np.unique(col[~np.isnan(col)])
    # Heuristic: categorical if <= 10 unique values and all are integers
    if len(unique_vals) <= 10 and np.all(np.equal(np.mod(unique_vals, 1), 0)):
        categorical_features.append(name)
    else:
        continuous_features.append(name)

print("Continuous features:", continuous_features)
print("Categorical features:", categorical_features)

Continuous features: ['WEIGHT2', 'HEIGHT3', 'ALCDAY5', 'FRUIT1', 'FVBEANS', 'FVGREEN', 'FVORANG', 'VEGETAB1', 'EXEROFT1', '_AGE80', 'HTIN4', 'HTM4', 'WTKG3', '_BMI5', '_DRNKWEK', '_FRUTSUM', '_VEGESUM', 'PA1MIN_']
Categorical features: ['GENHLTH', 'HLTHPLN1', 'MEDCOST', 'BPHIGH4', 'BPMEDS', 'TOLDHI2', 'CVDSTRK3', 'ASTHMA3', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY', 'DIABETE3', 'SEX', 'EDUCA', 'VETERAN3', 'EMPLOY1', 'INCOME2', 'SMOKE100', 'SMOKDAY2', 'LASTSMK2', 'USENOW3', 'EXERANY2', 'HIVTST6', 'CIMEMLOS', 'WTCHSALT', 'DRADVISE', 'HAREHAB1', 'STREHAB1', 'SCNTMNY1', 'SCNTMEL1', '_RFCHOL', '_LTASTH1', '_DRDXAR1', '_RACE', '_SMOKER3']


In [13]:
import pandas as pd

# Compute statistics for each feature in x_train_clean
stats = []
for i in range(x_train_clean.shape[1]):
    col = x_train_clean[:, i]
    col_nonan = col[~np.isnan(col)]
    stats.append({
        "Feature": feature_names[features_to_keep[i]],
        "Mean": np.mean(col_nonan),
        "Median": np.median(col_nonan),
        "Variance": np.var(col_nonan),
        "Min": np.percentile(col_nonan, 0),
        "Q1": np.percentile(col_nonan, 25),
        "Q2": np.percentile(col_nonan, 50),
        "Q3": np.percentile(col_nonan, 75),
        "Max": np.percentile(col_nonan, 100)
    })

stats_df = pd.DataFrame(stats)
stats_df.to_csv("feature_statistics.csv", index=False)

# Save as image
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, len(stats_df) * 0.3))
ax.axis('off')
table = ax.table(cellText=stats_df.values, colLabels=stats_df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.auto_set_column_width(col=list(range(len(stats_df.columns))))
plt.tight_layout()
plt.savefig("feature_statistics.png")
plt.close()

In [18]:
import os

import matplotlib.pyplot as plt

# Create output folder for histograms of relative frequency
hist_relfreq_dir = "hists_relfreq"
os.makedirs(hist_relfreq_dir, exist_ok=True)

for i in range(x_train_clean.shape[1]):
    feature_name = feature_names[features_to_keep[i]]
    col = x_train_clean[:, i]
    mask_neg1 = (y_train == -1) & (~np.isnan(col))
    mask_pos1 = (y_train == 1) & (~np.isnan(col))
    data_neg1 = col[mask_neg1]
    data_pos1 = col[mask_pos1]

    # Use bins based on unique values for categorical, or 30 bins for continuous
    if feature_name in categorical_features:
        bins = np.unique(np.concatenate([data_neg1, data_pos1]))
    else:
        bins = 30

    plt.figure(figsize=(8, 6))
    plt.hist(data_neg1, bins=bins, density=True, alpha=0.6, color='blue', label='y = -1', edgecolor='black')
    plt.hist(data_pos1, bins=bins, density=True, alpha=0.6, color='red', label='y = 1', edgecolor='black')
    plt.title(f"Relative Frequency Histogram of {feature_name} by y_train")
    plt.xlabel(feature_name)
    plt.ylabel("Relative Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(hist_relfreq_dir, f"{feature_name}_relfreq.png"))
    plt.close()

C:\Users\andre\AppData\Local\Temp\ipykernel_24812\1466216226.py:29: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
c:\Users\andre\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_histograms_impl.py:897: RuntimeWarning: invalid value encountered in divide
  return n / db / n.sum(), bin_edges
c:\Users\andre\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_histograms_impl.py:897: RuntimeWarning: invalid value encountered in divide
  return n / db / n.sum(), bin_edges


In [27]:
import os

import matplotlib.pyplot as plt

# Compute correlation matrix for x_train_clean (pairwise Pearson, ignoring NaNs)
corr_matrix = np.full((x_train_clean.shape[1], x_train_clean.shape[1]), np.nan)
for i in range(x_train_clean.shape[1]):
    for j in range(x_train_clean.shape[1]):
        col_i = x_train_clean[:, i]
        col_j = x_train_clean[:, j]
        mask = ~np.isnan(col_i) & ~np.isnan(col_j)
        if np.sum(mask) > 1 :
            std_i = np.std(col_i[mask])
            std_j = np.std(col_j[mask])
            if std_i == 0 or std_j == 0:
                corr_matrix[i,j] = np.nan
            else:
                corr_matrix[i,j] = np.corrcoef(col_i[mask], col_j[mask])[0,1]
np.fill_diagonal(corr_matrix, 1.0)

# Mark non-negligible correlations (absolute value > 0.3)
marked_005 = np.where(np.abs(corr_matrix) > 0.05, corr_matrix, np.nan)

# Plot and save
heatmap_dir = "heatmaps"
os.makedirs(heatmap_dir, exist_ok=True)

# Plot and save for |corr| > 0.05
plt.figure(figsize=(14, 12))
im = plt.imshow(marked_005, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.title("Correlation Matrix (|corr| > 0.05 marked)")
plt.xticks(range(len(features_to_keep)), [feature_names[i] for i in features_to_keep], rotation=90, fontsize=8)
plt.yticks(range(len(features_to_keep)), [feature_names[i] for i in features_to_keep], fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(heatmap_dir, "correlation_matrix_marked_005.png"))
plt.close()

# Subgroups: continuous and categorical features

# Other subgroups: define custom ranges of feature indices (by column in x_train_clean)
subgroups = [
    (0, 4),
    (2, 6),
    (4, 8),
    (6, 10),
    (8, 12),
    (10, 14),
    (12, 16),
    (14, 18),
    (16, 20),
    (18, 22),
    (20, 24),
    (22, 26),
    (24, 28),
    (26, 30),
    (28, 32),
    (30, 34),
    (32, 36),
    (34, 38),
    (36, 40),
    (38, 42),
    (40, 44),
    (42, 46),
    (44, 48),
    (46, 50),
    (48, 52),
    (50, 54)
]

for start, end in subgroups:
    subgroup_names = [feature_names[features_to_keep[i]] for i in range(start, end + 1)]
    corr_sub = corr_matrix[start:end+1, start:end+1]
    marked_sub = np.where(np.abs(corr_sub) > 0.05, corr_sub, np.nan)
    plt.figure(figsize=(6, 5))
    im = plt.imshow(marked_sub, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.title(f"Correlation Matrix (Features {start+1} to {end+1}, |corr| > 0.05 marked)")
    plt.xticks(range(len(subgroup_names)), subgroup_names, rotation=90, fontsize=8)
    plt.yticks(range(len(subgroup_names)), subgroup_names, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(heatmap_dir, f"correlation_matrix_subgroup_{start+1}_{end+1}.png"))
    plt.close()
continuous_idxs = [features_to_keep.index(feature_to_index[name]) for name in continuous_features if name in feature_to_index and feature_to_index[name] in features_to_keep]
categorical_idxs = [features_to_keep.index(feature_to_index[name]) for name in categorical_features if name in feature_to_index and feature_to_index[name] in features_to_keep]

# Continuous features correlation matrix
if continuous_idxs:
    corr_cont = corr_matrix[np.ix_(continuous_idxs, continuous_idxs)]
    marked_cont = np.where(np.abs(corr_cont) > 0.05, corr_cont, np.nan)
    plt.figure(figsize=(10, 8))
    im = plt.imshow(marked_cont, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.title("Correlation Matrix (Continuous Features, |corr| > 0.05 marked)")
    plt.xticks(range(len(continuous_idxs)), [continuous_features[i] for i in range(len(continuous_idxs))], rotation=90, fontsize=8)
    plt.yticks(range(len(continuous_idxs)), [continuous_features[i] for i in range(len(continuous_idxs))], fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(heatmap_dir, "correlation_matrix_continuous.png"))
    plt.close()

# Categorical features correlation matrix
if categorical_idxs:
    corr_cat = corr_matrix[np.ix_(categorical_idxs, categorical_idxs)]
    marked_cat = np.where(np.abs(corr_cat) > 0.05, corr_cat, np.nan)
    plt.figure(figsize=(10, 8))
    im = plt.imshow(marked_cat, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.title("Correlation Matrix (Categorical Features, |corr| > 0.05 marked)")
    plt.xticks(range(len(categorical_idxs)), [categorical_features[i] for i in range(len(categorical_idxs))], rotation=90, fontsize=8)
    plt.yticks(range(len(categorical_idxs)), [categorical_features[i] for i in range(len(categorical_idxs))], fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(heatmap_dir, "correlation_matrix_categorical.png"))
    plt.close()